# Eksploracja: naive_processor\n\nCo zwraca naiwna ekstrakcja tekstu i chunkowanie z PDF przez PyMuPDF?

In [48]:
import sys
from pathlib import Path

# dodaj root projektu do sys.path żeby importy działały
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "backend"))

from app.document.naive_processor import extract_pages, chunk_pages

## Ekstrakcja z mniejszego PDF

In [49]:
pdf_path = str(project_root / "data" / "raport_2024_wybrane_strony.pdf")
pages = extract_pages(pdf_path)
print(f"Zwrócono {len(pages)} stron")

Zwrócono 19 stron


## Struktura wyników

In [50]:
# struktura pojedynczego elementu
first = pages[0]
print(f"Klucze: {list(first.keys())}")
print(f"Typy:   text -> {type(first['text']).__name__}, page -> {type(first['page']).__name__}")
print(f"Numer strony: {first['page']}")
print(f"Długość tekstu: {len(first['text'])} znaków")

Klucze: ['text', 'page']
Typy:   text -> str, page -> int
Numer strony: 1
Długość tekstu: 499 znaków


In [51]:
# podgląd tekstu z pierwszej strony (pierwsze 500 znaków)
print(f"--- Strona {first['page']} (pierwsze 500 znaków) ---")
print(first["text"][:500])

--- Strona 1 (pierwsze 500 znaków) ---
Sprawozdanie Zarządu z działalności Grupy Kapitałowej
Banku Gospodarstwa Krajowego w 2024 roku
RAPORT ZINTEGROWANY
Obejmujące sprawozdanie Zarządu z działalności
Banku Gospodarstwa Krajowego
Dokument ten nie stanowi oficjalnej wersji sprawozdania. Oficjalne sprawozdanie Zarządu z działalności Grupy Kapitałowej Banku Gospodarstwa Krajowego w 2024 roku 
(obejmujące sprawozdanie Zarządu z działalności Banku Gospodarstwa Krajowego) - Raport zintegrowany zostało sporządzone zgodnie z wymogami ESEF.



In [52]:
# podgląd tekstu z ostatniej strony
last = pages[-1]
print(f"--- Strona {last['page']} (pierwsze 500 znaków) ---")
print(last["text"][:500])

--- Strona 19 (pierwsze 500 znaków) ---
Zapraszamy  
na stronę internetową  
naszego raportu
raportzintegrowany.bgk.pl/2024



## Statystyki długości tekstu per strona

In [53]:
lengths = [len(p["text"]) for p in pages]

print(f"Liczba stron z tekstem: {len(pages)}")
print(f"Min długość:  {min(lengths)} znaków")
print(f"Max długość:  {max(lengths)} znaków")
print(f"Średnia:      {sum(lengths) / len(lengths):.0f} znaków")
print(f"Suma:         {sum(lengths)} znaków")
print()
print("Długość per strona:")
for p, length in zip(pages, lengths):
    bar = "█" * (length // 100)
    print(f"  s.{p['page']:>3}: {length:>5} zn  {bar}")

Liczba stron z tekstem: 19
Min długość:  84 znaków
Max długość:  10052 znaków
Średnia:      2472 znaków
Suma:         46971 znaków

Długość per strona:
  s.  1:   499 zn  ████
  s.  2: 10052 zn  ████████████████████████████████████████████████████████████████████████████████████████████████████
  s.  3:   346 zn  ███
  s.  4:  3359 zn  █████████████████████████████████
  s.  5:  3187 zn  ███████████████████████████████
  s.  6:  1355 zn  █████████████
  s.  7:  1853 zn  ██████████████████
  s.  8:  3475 zn  ██████████████████████████████████
  s.  9:  2866 zn  ████████████████████████████
  s. 10:  1752 zn  █████████████████
  s. 11:   339 zn  ███
  s. 12:  2638 zn  ██████████████████████████
  s. 13:  2892 zn  ████████████████████████████
  s. 14:  3292 zn  ████████████████████████████████
  s. 15:   261 zn  ██
  s. 16:  3145 zn  ███████████████████████████████
  s. 17:  2631 zn  ██████████████████████████
  s. 18:  2945 zn  █████████████████████████████
  s. 19:    84 zn  


## Chunkowanie

In [54]:
chunks = chunk_pages(pages)
print(f"Z {len(pages)} stron powstało {len(chunks)} chunków (chunk_size=500, overlap=50)")

Z 19 stron powstało 114 chunków (chunk_size=500, overlap=50)


In [58]:
# podgląd pierwszych 3 chunków
for c in chunks[:3]:
    print(f"[strona {c['page']}, chunk {c['chunk_index']}] ({len(c['text'])} zn)")
    print(c["text"][:500])
    print("---")

[strona 1, chunk 0] (499 zn)
Sprawozdanie Zarządu z działalności Grupy Kapitałowej
Banku Gospodarstwa Krajowego w 2024 roku
RAPORT ZINTEGROWANY
Obejmujące sprawozdanie Zarządu z działalności
Banku Gospodarstwa Krajowego
Dokument ten nie stanowi oficjalnej wersji sprawozdania. Oficjalne sprawozdanie Zarządu z działalności Grupy Kapitałowej Banku Gospodarstwa Krajowego w 2024 roku 
(obejmujące sprawozdanie Zarządu z działalności Banku Gospodarstwa Krajowego) - Raport zintegrowany zostało sporządzone zgodnie z wymogami ESEF.

---
[strona 1, chunk 1] (49 zn)
any zostało sporządzone zgodnie z wymogami ESEF.

---
[strona 2, chunk 0] (500 zn)
2
Jedyny
taki bank
Tworzenie
wartości
Pracownicy
Otoczenie
Osiągnięcia
Ład korporacyjny
Zarządzanie 
ryzykiem
Perspektywy
Raportowanie
Załączniki
Spis treści
I 
Jedyny taki bank 
3
1. List Prezesa Zarządu .......................................................................................................................................................

In [56]:
# statystyki chunków
chunk_lengths = [len(c["text"]) for c in chunks]
print(f"Liczba chunków: {len(chunks)}")
print(f"Min: {min(chunk_lengths)}, Max: {max(chunk_lengths)}, Średnia: {sum(chunk_lengths)/len(chunk_lengths):.0f}")
print(f"Chunki po 500 zn: {sum(1 for l in chunk_lengths if l == 500)}")
print(f"Chunki krótsze:   {sum(1 for l in chunk_lengths if l < 500)}")

Liczba chunków: 114
Min: 5, Max: 500, Średnia: 453
Chunki po 500 zn: 92
Chunki krótsze:   22


## Porównanie: pełny raport

In [57]:
full_pdf_path = str(project_root / "data" / "raport_2024_pl 1.pdf")
full_pages = extract_pages(full_pdf_path)

full_lengths = [len(p["text"]) for p in full_pages]
print(f"Pełny raport: {len(full_pages)} stron z tekstem")
print(f"Min: {min(full_lengths)}, Max: {max(full_lengths)}, Średnia: {sum(full_lengths)/len(full_lengths):.0f}")
print(f"Łączny tekst: {sum(full_lengths)} znaków ({sum(full_lengths)/1000:.1f}k)")

Pełny raport: 184 stron z tekstem
Min: 84, Max: 10052, Średnia: 2978
Łączny tekst: 548027 znaków (548.0k)
